# Columnar Databases — First Contact

Instead of storing all columns for a row together, columnar databases store all values for one column together. For analytics that read 2 columns out of 50, you only touch 2 columns worth of data — the other 48 never leave disk. DuckDB is the local columnar engine: no server, pure Python, blazing fast. It attaches directly to Postgres, Parquet files, CSVs, or S3 — and runs vectorized SQL against any of them.

## What makes columnar different

- **Columnar storage** — only reads the columns you ask for; skips the rest entirely at the storage layer.
- **Vectorized execution** — processes batches of values (vectors) at once using SIMD CPU instructions, not row by row.
- **Compression** — same-type values (all ints, all timestamps) compress far better than mixed-type rows; less I/O = faster queries.
- **When to use** — analytical queries, aggregations, GROUP BY / ORDER BY on wide tables, read-heavy workloads where you rarely need all columns.

In [1]:
from pathlib import Path
import sys

# Locate _setup whether Jupyter CWD is the notebook folder or workspace root
for _candidate in [Path('_setup'), Path('Basics/Databases/_setup')]:
    if _candidate.exists():
        sys.path.insert(0, str(_candidate.resolve()))
        break

import pandas as pd
import time
import os
from db_connections import get_postgres_conn, get_duckdb_conn

pg_conn = get_postgres_conn()
duck = get_duckdb_conn()   # already attaches pg internally — we'll re-attach explicitly in next cell
print("Both connected.")

Both connected.


## The same 5 queries — DuckDB vs Postgres

DuckDB attaches to Postgres over a local socket and runs the same queries. We time both engines on identical SQL and compare. Note: DuckDB over an attached Postgres is **not** its true home — it still has to pull data through the Postgres wire protocol. DuckDB shines hardest on local Parquet files (Round 2). This is just the first look.

In [2]:
# Detach pg if get_duckdb_conn() already attached it, then re-attach explicitly
try:
    duck.execute("DETACH pg")
except Exception:
    pass

duck.execute("INSTALL postgres; LOAD postgres;")

pg_dsn = (
    f"host=localhost "
    f"port={os.getenv('POSTGRES_PORT', 5432)} "
    f"dbname={os.getenv('POSTGRES_DB')} "
    f"user={os.getenv('POSTGRES_USER')} "
    f"password={os.getenv('POSTGRES_PASSWORD')}"
)
duck.execute(f"ATTACH '{pg_dsn}' AS pg (TYPE postgres, READ_ONLY);")

dbs = duck.execute("SHOW DATABASES").df()
print("Postgres attached to DuckDB.")
print(dbs)

Postgres attached to DuckDB.
  database_name
0        memory
1            pg


In [3]:
# Query 1 — endpoints by datacenter
q1_pg  = "SELECT datacenter, COUNT(*) AS endpoint_count FROM telemetry.endpoints GROUP BY datacenter ORDER BY endpoint_count DESC"
q1_duck = "SELECT datacenter, COUNT(*) AS endpoint_count FROM pg.telemetry.endpoints GROUP BY datacenter ORDER BY endpoint_count DESC"

t0 = time.perf_counter()
df1 = duck.execute(q1_duck).df()
duck_ms = (time.perf_counter() - t0) * 1000

print(f"DuckDB: {duck_ms:.1f}ms")
df1

DuckDB: 39.6ms


,datacenter,endpoint_count
0,SNG1,2579
1,NYC1,2489
2,LON1,2485
3,NYC2,2447


In [4]:
# Query 2 — top 10 most alerted endpoints
q2_duck = """
SELECT
    e.hostname,
    e.datacenter,
    e.service_type,
    COUNT(a.alert_id) AS alert_count
FROM pg.telemetry.endpoints e
JOIN pg.telemetry.alerts a ON a.endpoint_id = e.endpoint_id
GROUP BY e.endpoint_id, e.hostname, e.datacenter, e.service_type
ORDER BY alert_count DESC
LIMIT 10
"""

t0 = time.perf_counter()
df2 = duck.execute(q2_duck).df()
duck_ms = (time.perf_counter() - t0) * 1000

print(f"DuckDB: {duck_ms:.1f}ms")
df2

DuckDB: 22.0ms


,hostname,datacenter,service_type,alert_count
0,srv-04452.citi.internal,NYC1,worker,12
1,srv-07578.citi.internal,NYC1,web,11
2,srv-03610.citi.internal,NYC2,cache,10
3,srv-02253.citi.internal,SNG1,worker,10
4,srv-03423.citi.internal,NYC1,monitor,10
5,srv-01617.citi.internal,SNG1,db,10
6,srv-01000.citi.internal,LON1,cache,9
7,srv-08214.citi.internal,NYC1,worker,9
8,srv-01648.citi.internal,NYC1,db,9
9,srv-01152.citi.internal,SNG1,db,9


In [5]:
# Query 3 — avg CPU percent by service_type, last 7 days
q3_duck = """
SELECT
    e.service_type,
    ROUND(AVG(m.value), 2) AS avg_cpu_percent,
    COUNT(*) AS sample_count
FROM pg.telemetry.metrics m
JOIN pg.telemetry.endpoints e ON e.endpoint_id = m.endpoint_id
WHERE
    m.metric_name = 'cpu_percent'
    AND m.recorded_at >= NOW() - INTERVAL '7 days'
GROUP BY e.service_type
ORDER BY avg_cpu_percent DESC
"""

t0 = time.perf_counter()
df3 = duck.execute(q3_duck).df()
duck_ms = (time.perf_counter() - t0) * 1000

print(f"DuckDB: {duck_ms:.1f}ms")
df3

DuckDB: 388.2ms


,service_type,avg_cpu_percent,sample_count
0,worker,52.82,1535
1,db,52.62,1410
2,monitor,52.60,1358
3,cache,52.17,1428
4,web,52.05,1432


In [6]:
# Query 4 — open critical alerts with endpoint details
q4_duck = """
SELECT
    a.alert_id,
    e.hostname,
    e.datacenter,
    e.service_type,
    a.category,
    a.message,
    a.created_at
FROM pg.telemetry.alerts a
JOIN pg.telemetry.endpoints e ON e.endpoint_id = a.endpoint_id
WHERE
    a.severity = 'critical'
    AND a.status = 'open'
ORDER BY a.created_at DESC
LIMIT 20
"""

t0 = time.perf_counter()
df4 = duck.execute(q4_duck).df()
duck_ms = (time.perf_counter() - t0) * 1000

print(f"DuckDB: {duck_ms:.1f}ms — {len(df4)} rows")
df4

DuckDB: 94.0ms — 20 rows


,alert_id,hostname,datacenter,service_type,category,message,created_at
0,c86c8fc0-3625-4453-8a2e-f3f824befaed,srv-03961.citi.internal,NYC1,worker,network,Load average exceeded 4x CPU count,2026-03-23 06:09:58.063609-05:00
1,37b85a13-8780-4188-b244-94e73812f771,srv-05185.citi.internal,SNG1,web,cpu,Network throughput dropped below SLA threshold,2026-03-23 05:59:08.063609-05:00
2,e96361a9-f106-4faf-89c1-f81c8d0d976a,srv-09205.citi.internal,NYC2,monitor,disk,CPU utilization exceeded 90% threshold for 5 m...,2026-03-23 05:51:38.063609-05:00
3,a937e784-5b1f-4ca1-bbdc-90b6463decc1,srv-08492.citi.internal,SNG1,worker,memory,TCP connection pool exhausted on port 5432,2026-03-23 05:32:23.063609-05:00
4,f00672ee-30f6-4547-9dcd-83ee46ddae60,srv-03158.citi.internal,LON1,monitor,network,SSL certificate expires in 7 days,2026-03-23 04:38:57.063609-05:00
5,1c6b33a1-f776-497e-9c8a-96d0327c49bb,srv-09260.citi.internal,NYC1,db,disk,Memory usage at 95% — potential OOM imminent,2026-03-23 03:52:47.063609-05:00
6,8c1496c9-3fb9-4e43-b885-bbde555de28d,srv-01438.citi.internal,SNG1,web,disk,Disk I/O latency spike detected: 450ms avg,2026-03-23 03:33:59.063609-05:00
7,2ee1ac9b-614d-486b-a44a-f13255990e90,srv-01191.citi.internal,NYC1,web,disk,Process georgenichols exited unexpectedly — re...,2026-03-23 03:05:28.063609-05:00
8,6156565c-da0d-43e8-acd7-d8a94c503f8e,srv-06141.citi.internal,NYC1,monitor,network,CPU utilization exceeded 90% threshold for 5 m...,2026-03-23 01:48:31.063609-05:00
9,f0c24621-26a8-466e-a9a1-9436cefc72c3,srv-00057.citi.internal,LON1,db,network,TCP connection pool exhausted on port 5432,2026-03-23 01:42:24.063609-05:00


In [7]:
# Query 5 — side-by-side timing comparison
queries = [
    (
        "Q1 endpoints by DC",
        "SELECT datacenter, COUNT(*) AS endpoint_count FROM telemetry.endpoints GROUP BY datacenter ORDER BY endpoint_count DESC",
        "SELECT datacenter, COUNT(*) AS endpoint_count FROM pg.telemetry.endpoints GROUP BY datacenter ORDER BY endpoint_count DESC",
    ),
    (
        "Q2 top 10 alerted",
        """SELECT e.hostname, COUNT(a.alert_id) AS alert_count
           FROM telemetry.endpoints e JOIN telemetry.alerts a ON a.endpoint_id = e.endpoint_id
           GROUP BY e.endpoint_id, e.hostname ORDER BY alert_count DESC LIMIT 10""",
        """SELECT e.hostname, COUNT(a.alert_id) AS alert_count
           FROM pg.telemetry.endpoints e JOIN pg.telemetry.alerts a ON a.endpoint_id = e.endpoint_id
           GROUP BY e.endpoint_id, e.hostname ORDER BY alert_count DESC LIMIT 10""",
    ),
    (
        "Q3 avg CPU/service",
        """SELECT e.service_type, ROUND(AVG(m.value)::numeric, 2) AS avg_cpu
           FROM telemetry.metrics m JOIN telemetry.endpoints e ON e.endpoint_id = m.endpoint_id
           WHERE m.metric_name = 'cpu_percent' AND m.recorded_at >= NOW() - INTERVAL '7 days'
           GROUP BY e.service_type ORDER BY avg_cpu DESC""",
        """SELECT e.service_type, ROUND(AVG(m.value), 2) AS avg_cpu
           FROM pg.telemetry.metrics m JOIN pg.telemetry.endpoints e ON e.endpoint_id = m.endpoint_id
           WHERE m.metric_name = 'cpu_percent' AND m.recorded_at >= NOW() - INTERVAL '7 days'
           GROUP BY e.service_type ORDER BY avg_cpu DESC""",
    ),
    (
        "Q4 open critical alerts",
        """SELECT a.alert_id, e.hostname FROM telemetry.alerts a
           JOIN telemetry.endpoints e ON e.endpoint_id = a.endpoint_id
           WHERE a.severity = 'critical' AND a.status = 'open'
           ORDER BY a.created_at DESC LIMIT 20""",
        """SELECT a.alert_id, e.hostname FROM pg.telemetry.alerts a
           JOIN pg.telemetry.endpoints e ON e.endpoint_id = a.endpoint_id
           WHERE a.severity = 'critical' AND a.status = 'open'
           ORDER BY a.created_at DESC LIMIT 20""",
    ),
]

rows = []
for label, pg_sql, duck_sql in queries:
    t0 = time.perf_counter()
    pd.read_sql(pg_sql, pg_conn)
    pg_ms = (time.perf_counter() - t0) * 1000

    t0 = time.perf_counter()
    duck.execute(duck_sql).df()
    dk_ms = (time.perf_counter() - t0) * 1000

    winner = "DuckDB" if dk_ms < pg_ms else "Postgres"
    rows.append({"Query": label, "Postgres (ms)": round(pg_ms, 1), "DuckDB (ms)": round(dk_ms, 1), "Winner": winner})

comparison = pd.DataFrame(rows)
print(comparison.to_string(index=False))

C:\Users\shareuser\AppData\Local\Temp\ipykernel_30860\3350366118.py:44: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  pd.read_sql(pg_sql, pg_conn)


                  Query  Postgres (ms)  DuckDB (ms)   Winner
     Q1 endpoints by DC          145.8          4.2   DuckDB
      Q2 top 10 alerted           11.1         15.9 Postgres
     Q3 avg CPU/service           34.8        221.3 Postgres
Q4 open critical alerts            5.3         18.4 Postgres


## Key observations

- **DuckDB over attached Postgres is not its true home** — it still pulls data through the Postgres wire protocol, so the speedup (if any) is modest here.
- **Where DuckDB wins** — aggregation-heavy queries (Q1, Q3) benefit from vectorized execution even when data comes over a socket. _Fill in actual ms after running._
- **Where Postgres wins** — filtered lookups with indexes (Q4) favour Postgres's native index structures. DuckDB has to scan the full pull. _Fill in after running._
- **The real DuckDB showcase** — query a local Parquet file with 500K rows; DuckDB typically beats Postgres 5–20× because it reads only the columns needed, fully in-process, with no network. That's Round 2.
- **Numbers to record** — paste the comparison table output here after running. _Fill in._